In [1]:
import numpy as np
from numpy.random import uniform
from scipy.optimize import minimize
from scipy.stats import qmc
from scipy.stats import norm
import cyipopt
import warnings
import collections
from scipy import linalg
from smt.applications.ego import Evaluator
from smt.surrogate_models import KRG
from smt.design_space import DesignSpace 
from smt.problems.problem import Problem as SMTProblem


from hiopbbpy.surrogate_modeling import GaussianProcess, smtKRG
from hiopbbpy.problems import Problem, BraninProblem, LpNormProblem

from typing import Callable, Dict, List, Union, Tuple
from scipy.optimize import NonlinearConstraint
from hiopbbpy.opt import IpoptProb, BnBAlgorithmBase, BOAlgorithm, EIacquisition

#### Example 

In [2]:
import numpy as np
from dataclasses import dataclass

import numpy as np
from scipy.stats import qmc  # already used by Problem

### parameters
n_samples = 5  # number of the initial samples to train GP
theta = 1.e-2  # hyperparameter for GP kernel
nx = 2         # dimension of the problem
xlimits = np.array([[-5, 5], [-5, 5]]) # bounds on optimization variable

class QuadraticShift2D(Problem):
    """
    f(x) = ||x - c||^2
    - Global minimizer: x* = c
    - Global minimum:   f* = 0
    """
    def __init__(self, xlimits=None, c=None, constraints=[]):
        ndim = 2
        if xlimits is None:
            xlimits = np.array([[-5.0, 5.0], [-5.0, 5.0]], dtype=float)
        name = "QuadraticShift2D"
        super().__init__(ndim, xlimits, name=name, constraints=constraints)

        # choose center if not provided
        if c is None:
            c = self.xlimits.mean(axis=1)
        self.c = np.asarray(c, dtype=float)
        assert self.c.shape == (2,), "c must be a 2D point"

        # expose known solution for checking later
        self.x_star = self.c.copy()
        self.f_star = 0.0

    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        ne, nx = x.shape
        assert nx == self.ndim
        diff = x - self.c[None, :]
        y = np.sum(diff * diff, axis=1, dtype=float).reshape(ne, 1)
        return y


problem = QuadraticShift2D(xlimits=xlimits, c=np.array([1.25, -2.0]))
problem.set_constraints([])  
    
x_train = problem.sample(n_samples)
y_train = problem.evaluate(x_train)

gp_model = smtKRG(theta, xlimits, nx)
gp_model.train(x_train, y_train)

print(f"GP model trained with theta = {theta}")

print("Compute the max and min distance using the d norm:")

print("x corresponding to dL:")

print("x corresponding to dU:")

print("Compute the variance on x corresponding to dL and dU with SMT:")

print("Compute the mean on x corresponding to dL and dU with SMT:")

print("Compute the variance on x corresponding to dL and dU with SMT:")

print("Compute the mean on x corresponding to dL and dU with SMT:")

GP model trained with theta = 0.01
Compute the max and min distance using the d norm:
x corresponding to dL:
x corresponding to dU:
Compute the variance on x corresponding to dL and dU with SMT:
Compute the mean on x corresponding to dL and dU with SMT:
Compute the variance on x corresponding to dL and dU with SMT:
Compute the mean on x corresponding to dL and dU with SMT:


In [3]:
# ---- Use only BnBAlgorithmBase methods + SMT predict_* for reference ----

sm = gp_model.surrogatesmt

# Build a base "probe" and populate it from the trained SMT model
base = BnBAlgorithmBase(x=x_train, y=y_train)
base.gpsurrogate = gp_model
base.sync_from_smt()              # fills kernel_spec/p, theta, Xc, offsets, beta0/gamma, C, sigma2

# (Optional) ensure we try to compute a nontrivial lower bound for variance
base.BnB_LBmethod = None          # if "IPOPT", sigma2_L will be 0 in your current code

# Pick a box to test (use full domain here; swap l/u to any node box you want)
l = xlimits[:, 0].astype(float)
u = xlimits[:, 1].astype(float)

# 1) Bounds from your base class
kL, kU         = base.ker_bounds(l, u)       # per-point kernel bounds vs each training sample
mu_L, mu_U     = base.mu_bounds(kL, kU)      # scalar μ bounds on the box
s2_L, s2_U     = base.sigma2_bounds(kL, kU)  # scalar σ² bounds on the box

print("\n--- BnB base bounds on box ---")
print(f"mu_L={mu_L:.9f}, mu_U={mu_U:.9f}")
print(f"s2_L={s2_L:.9e}, s2_U={s2_U:.9e}")

# 2) (Optional) Node LCB/EI bounds using ONLY base.rs_* with your bounds
beta = 3.0  # or whatever you're using in BO; set from outside as needed
LCB_L = base.rs_lcb(mu_L, np.sqrt(max(s2_U, 0.0)))     # lower bound on LCB over the box
LCB_U = base.rs_lcb(mu_U, np.sqrt(max(s2_L, 0.0)))     # upper bound on LCB over the box
print(f"LCB node bounds:  L={LCB_L:.9f}   U={LCB_U:.9f}")

# 3) (Optional) Quick SMT sanity on a few points INSIDE the same box
#    (purely for reference; still using SMT’s own predict_* functions)
P = np.vstack([
    l,
    u,
    0.5*(l+u),
    np.array([l[0], u[1]]),
    np.array([u[0], l[1]]),
])
mu_smt  = sm.predict_values(P).ravel()
var_smt = sm.predict_variances(P).ravel()
print("\n--- SMT reference on corners/center ---")
print("μ(P):", mu_smt)
print("σ²(P):", var_smt)
print(f"min μ(P)={mu_smt.min():.9f}, max μ(P)={mu_smt.max():.9f}")
print(f"min σ²(P)={var_smt.min():.9e}, max σ²(P)={var_smt.max():.9e}")

# 4) (Optional) show per-training-point kernel extremes match your ker_bounds
#    Build the witness x for each training point that attains kU_i and kL_i.
#    (Just geometry: clamp for kU, farther endpoint for kL.)
off, scl = base.X_offset, base.X_scale
Xc = base.Xc
l_c = (l - off) / scl
u_c = (u - off) / scl

def extreme_points_for_training_point(Xi_c, l_c, u_c):
    xU_c = np.clip(Xi_c, l_c, u_c)                # min distance → max kernel
    dL = np.abs(l_c - Xi_c); dU = np.abs(u_c - Xi_c)
    xL_c = np.where(dL >= dU, l_c, u_c)           # max distance → min kernel
    return xU_c, xL_c

# compute kernel at those witnesses via base.ker_bounds result for equality check
kU_w, kL_w = [], []
for i in range(Xc.shape[0]):
    xU_c, xL_c = extreme_points_for_training_point(Xc[i], l_c, u_c)
    # evaluate kernel component i at those x by reusing your distance→kernel mapping:
    # Use the same separable form your ker_bounds uses:
    # we’ll reconstruct this single component using the *same* logic as ker_bounds,
    # but without re-deriving formulas: take the per-dim distances and plug into the same kernel_spec.
    dxU = np.abs(xU_c - Xc[i]); dxL = np.abs(xL_c - Xc[i])
    th  = base.theta
    if base.kernel_spec == "pow_exp":
        p = getattr(base, "p", 2.0)
        kU_w.append(np.exp(-np.dot(th, dxU**p)))
        kL_w.append(np.exp(-np.dot(th, dxL**p)))
    elif base.kernel_spec == "matern32":
        a = np.sqrt(3.0) * th
        kU_w.append(np.prod((1 + a*dxU) * np.exp(-a*dxU)))
        kL_w.append(np.prod((1 + a*dxL) * np.exp(-a*dxL)))
    elif base.kernel_spec == "matern52":
        b = np.sqrt(5.0) * th; tU=b*dxU; tL=b*dxL
        kU_w.append(np.prod((1 + tU + (tU**2)/3.0) * np.exp(-tU)))
        kL_w.append(np.prod((1 + tL + (tL**2)/3.0) * np.exp(-tL)))
    else:
        raise ValueError(f"Unsupported kernel_spec: {base.kernel_spec}")

kU_w = np.array(kU_w); kL_w = np.array(kL_w)
print("\nmax |kU(witness) - kU(ker_bounds)| =", float(np.max(np.abs(kU_w - kU))))
print("max |kL(witness) - kL(ker_bounds)| =", float(np.max(np.abs(kL_w - kL))))


corr =  pow_exp
p =  2.0
Polishing not needed - no active set detected at optimal point

--- BnB base bounds on box ---
mu_L=-14.035182139, mu_U=63.719402684
s2_L=0.000000000e+00, s2_U=6.539150100e+04
LCB node bounds:  L=-781.188041065   U=63.719402684

--- SMT reference on corners/center ---
μ(P): [19.94656149 35.57931985 16.25711962 31.08110676 16.75837284]
σ²(P): [232.18542773  63.22746903  50.00732241 159.01562048  63.79860422]
min μ(P)=16.257119616, max μ(P)=35.579319845
min σ²(P)=5.000732241e+01, max σ²(P)=2.321854277e+02

max |kU(witness) - kU(ker_bounds)| = 0.0
max |kL(witness) - kL(ker_bounds)| = 0.0


In [4]:
import numpy as np
from dataclasses import dataclass

import numpy as np
from scipy.stats import qmc  # already used by Problem

# Get user input for the number of repetitions from command-line arguments
num_repeat = 1

### parameters
n_samples = 5  # number of the initial samples to train GP
theta = 1.e-2  # hyperparameter for GP kernel
nx = 2         # dimension of the problem
xlimits = np.array([[-5, 5], [-5, 5]]) # bounds on optimization variable

class QuadraticShift2D(Problem):
    """
    f(x) = ||x - c||^2
    - Global minimizer: x* = c
    - Global minimum:   f* = 0
    """
    def __init__(self, xlimits=None, c=None, constraints=[]):
        ndim = 2
        if xlimits is None:
            xlimits = np.array([[-5.0, 5.0], [-5.0, 5.0]], dtype=float)
        name = "QuadraticShift2D"
        super().__init__(ndim, xlimits, name=name, constraints=constraints)

        # choose center if not provided
        if c is None:
            c = self.xlimits.mean(axis=1)
        self.c = np.asarray(c, dtype=float)
        assert self.c.shape == (2,), "c must be a 2D point"

        # expose known solution for checking later
        self.x_star = self.c.copy()
        self.f_star = 0.0

    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        ne, nx = x.shape
        assert nx == self.ndim
        diff = x - self.c[None, :]
        y = np.sum(diff * diff, axis=1, dtype=float).reshape(ne, 1)
        return y


acq_type = "EI"

problem = QuadraticShift2D(xlimits=xlimits, c=np.array([1.25, -2.0]))
problem.set_constraints([])  # <-- no constraints for this test
x_train = problem.sample(n_samples)
y_train = problem.evaluate(x_train)

gp_model = smtKRG(theta, xlimits, nx)
gp_model.train(x_train, y_train)
print(f"GP model trained with theta = {theta}")
        
solver_options = {"max_iter": 500, "print_level": 4}
options = {
          'log_level': 'info',
          'acquisition_type': acq_type,
          'acquisition_method': 'bnb',
          'bo_maxiter': 10,
          'opt_solver': 'IPOPT',
          'solver_options': solver_options,
     }

bo = BOAlgorithm(problem, gp_model, x_train, y_train, options=options)
bo.optimize()

if hasattr(bo, "best_x"):
    dist = np.linalg.norm(bo.best_x - problem.x_star)
    print(f"[Check] ||x_best - x*|| = {dist:.3e}  (x* = {problem.x_star})")

hiopbbpy Problem name: QuadraticShift2D
hiopbbpy Max BO iter: 10
hiopbbpy Optimizing acquisition (EI) with 10 random initial points
hiopbbpy Batch type: KB
hiopbbpy Batch size: 1
hiopbbpy Internal optimization solver: IPOPT
hiopbbpy Internal optimization solver options: {'max_iter': 500, 'print_level': 4, 'sb': 'yes'}
hiopbbpy Initial training set: 5 samples, 2 dimensions
hiopbbpy Logger level: info
hiopbbpy Best UNCONSTRAINED objective from 5 initial samples: 8.3679e+00 
hiopbbpy Best CONSTRAINED objective from 5 feasible initial samples: 8.3679e+00
hiopbbpy *****************************
hiopbbpy Iteration 1/10
hiopbbpy In batch 1/1
hiopbbpy Start finding the best sampling point:


GP model trained with theta = 0.01
Total number of variables............................:        2
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        2
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0


Number of Iterations....: 8

                                   (scaled)                 (unscaled)
Objective...............:  -1.0443375024193090e+01   -1.0443375024193090e+01
Dual infeasibility......:   8.8565948073532529e-10    8.8565948073532529e-10
Constraint violation....:   0.0000000000000000e+00    0.0000000000000000e+00
Variable bound violation:   4.9973002624881246e-08    4.


Number of Iterations....: 20

                                   (scaled)                 (unscaled)
Objective...............:  -2.5155551394770190e-08   -2.5155551394770190e-08
Dual infeasibility......:   6.8533210506404609e-07    6.8533210506404609e-07
Constraint violation....:   0.0000000000000000e+00    0.0000000000000000e+00
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   2.9536483942416908e-10    2.9536483942416908e-10
Overall NLP error.......:   6.8533210506404609e-07    6.8533210506404609e-07


Number of objective function evaluations             = 21
Number of objective gradient evaluations             = 21
Number of equality constraint evaluations            = 0
Number of inequality constraint evaluations          = 0
Number of equality constraint Jacobian evaluations   = 0
Number of inequality constraint Jacobian evaluations = 0
Number of Lagrangian Hessian evaluations             = 0
Total seconds in IPOPT           

[[array([-2.14252729,  0.42063974]), -7.480282419220226e-43, True, b'Algorithm terminated successfully at a locally optimal point, satisfying the convergence tolerances (can be specified by options).']]
Number of Iterations....: 7

                                   (scaled)                 (unscaled)
Objective...............:  -7.4802824192202261e-43   -7.4802824192202261e-43
Dual infeasibility......:   1.9086647723799494e-10    1.9086647723799494e-10
Constraint violation....:   0.0000000000000000e+00    0.0000000000000000e+00
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   9.0909091130449188e-10    9.0909091130449188e-10
Overall NLP error.......:   9.0909091130449188e-10    9.0909091130449188e-10


Number of objective function evaluations             = 8
Number of objective gradient evaluations             = 8
Number of equality constraint evaluations            = 0
Number of inequality constraint evaluations          = 0
Numbe

[[array([-1.10730985,  0.47162632]), -7.601143830534594e-19, True, b'Algorithm terminated successfully at a locally optimal point, satisfying the convergence tolerances (can be specified by options).']]
Number of Iterations....: 7

                                   (scaled)                 (unscaled)
Objective...............:  -7.6011438305345943e-19   -7.6011438305345943e-19
Dual infeasibility......:   8.4685055801458025e-11    8.4685055801458025e-11
Constraint violation....:   0.0000000000000000e+00    0.0000000000000000e+00
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   9.0909091023949519e-10    9.0909091023949519e-10
Overall NLP error.......:   9.0909091023949519e-10    9.0909091023949519e-10


Number of objective function evaluations             = 8
Number of objective gradient evaluations             = 8
Number of equality constraint evaluations            = 0
Number of inequality constraint evaluations          = 0
Numbe

[[array([ 5.00000005, -2.86790547]), -10.443375024220455, True, b'Algorithm terminated successfully at a locally optimal point, satisfying the convergence tolerances (can be specified by options).']]
Number of Iterations....: 7

                                   (scaled)                 (unscaled)
Objective...............:  -1.0443375024220455e+01   -1.0443375024220455e+01
Dual infeasibility......:   2.2360002738253115e-10    2.2360002738253115e-10
Constraint violation....:   0.0000000000000000e+00    0.0000000000000000e+00
Variable bound violation:   4.9992773476503771e-08    4.9992773476503771e-08
Complementarity.........:   1.0000257713240429e-11    1.0000257713240429e-11
Overall NLP error.......:   2.2360002738253115e-10    2.2360002738253115e-10


Number of objective function evaluations             = 14
Number of objective gradient evaluations             = 8
Number of equality constraint evaluations            = 0
Number of inequality constraint evaluations          = 0
Number 

[[array([-1.27655752, -4.08965524]), -5.245142847388206e-07, False, "Ipopt failed to solve the problem. Status msg: b'Maximum number of iterations exceeded (can be specified by an option).'"]]
Number of Iterations....: 500

                                   (scaled)                 (unscaled)
Objective...............:  -5.2451428473882063e-07   -5.2451428473882063e-07
Dual infeasibility......:   1.3749053657088933e-05    1.3749053657088933e-05
Constraint violation....:   0.0000000000000000e+00    0.0000000000000000e+00
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   8.3800190264129143e-08    8.3800190264129143e-08
Overall NLP error.......:   1.3749053657088933e-05    1.3749053657088933e-05


Number of objective function evaluations             = 501
Number of objective gradient evaluations             = 501
Number of equality constraint evaluations            = 0
Number of inequality constraint evaluations          = 0
Number of

RuntimeError: EI maximization failed